# FixtureDB: Fixture Complexity Analysis

This notebook examines the structural characteristics of test fixtures:
- **Fixture count**: How many fixtures per repository
- **Fixture scope**: Distribution across module, class, and function scope
- **Nesting depth**: Code structure complexity (tree-sitter AST analysis)
- **Cognitive complexity**: Relationship with nesting depth

These metrics reveal patterns in how developers structure fixture code.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

plt.rcParams['figure.facecolor'] = '#FAFAFA'
print("✓ Libraries imported")

In [ ]:
data_dir = Path('fixturedb')
repos = pd.read_csv(data_dir / 'repositories.csv')
fixtures = pd.read_csv(data_dir / 'fixtures.csv')

print(f"Repositories: {len(repos)}")
print(f"Fixtures: {len(fixtures)}")
print(f"\nFixture columns: {', '.join(fixtures.columns.tolist())}")

## Fixture Count Distribution

In [ ]:
print("Fixtures per Repository:")
print(repos['num_fixtures'].describe())

fig, ax = plt.subplots(figsize=(10, 6), facecolor='#FAFAFA')
repos['num_fixtures'].hist(bins=50, ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Distribution of Fixtures per Repository', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Fixtures', fontsize=12)
ax.set_ylabel('Number of Repositories', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Fixture Scope Distribution

In [ ]:
if 'scope' in fixtures.columns:
    scope_counts = fixtures['scope'].value_counts()
    print("Fixtures by Scope:")
    print(scope_counts)
    
    fig, ax = plt.subplots(figsize=(10, 6), facecolor='#FAFAFA')
    scope_counts.plot(kind='bar', ax=ax, color=sns.color_palette('Set2', len(scope_counts)))
    ax.set_title('Fixture Scope Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Scope', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("'scope' column not available in fixtures.csv")

## Nesting Depth Analysis

In [ ]:
if 'max_nesting_depth' in fixtures.columns:
    print("Max Nesting Depth Statistics:")
    print(fixtures['max_nesting_depth'].describe())
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 5), facecolor='#FAFAFA')
    
    # Histogram
    fixtures['max_nesting_depth'].hist(bins=30, ax=axs[0], color='coral', edgecolor='black')
    axs[0].set_title('Nesting Depth Distribution', fontsize=12, fontweight='bold')
    axs[0].set_xlabel('Max Nesting Depth', fontsize=11)
    axs[0].set_ylabel('Count', fontsize=11)
    axs[0].grid(axis='y', alpha=0.3)
    
    # Box plot by language
    fixtures_with_lang = fixtures.merge(repos[['id', 'language']], left_on='repo_id', right_on='id', how='left')
    fixtures_with_lang.boxplot(column='max_nesting_depth', by='language', ax=axs[1])
    axs[1].set_title('Nesting Depth by Language', fontsize=12, fontweight='bold')
    axs[1].set_xlabel('Language', fontsize=11)
    axs[1].set_ylabel('Max Nesting Depth', fontsize=11)
    plt.suptitle('')  # Remove default title
    
    plt.tight_layout()
    plt.show()
else:
    print("'max_nesting_depth' column not available")

## Nesting Depth vs Cognitive Complexity

In [ ]:
if 'max_nesting_depth' in fixtures.columns and 'cognitive_complexity' in fixtures.columns:
    fig, ax = plt.subplots(figsize=(10, 6), facecolor='#FAFAFA')
    
    ax.scatter(fixtures['max_nesting_depth'], fixtures['cognitive_complexity'], 
               alpha=0.5, s=30, color='steelblue', edgecolors='black', linewidth=0.5)
    ax.set_title('Nesting Depth vs Cognitive Complexity', fontsize=14, fontweight='bold')
    ax.set_xlabel('Max Nesting Depth', fontsize=12)
    ax.set_ylabel('Cognitive Complexity', fontsize=12)
    ax.grid(alpha=0.3)
    
    # Calculate correlation
    corr = fixtures['max_nesting_depth'].corr(fixtures['cognitive_complexity'])
    ax.text(0.95, 0.05, f'Pearson r = {corr:.3f}', 
            transform=ax.transAxes, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
else:
    print("Required columns not available")

## Key Findings

**Fixture Complexity Patterns:**
- Fixture counts vary widely across repositories
- Scope distribution reveals preferred patterns (module, class, function)
- Nesting depth shows most fixtures are relatively simple
- Moderate correlation between nesting depth and cognitive complexity validates both metrics